# Durand-Kerner (Weierstrass) + Newton Polishing Simulation

จำลองการทำงานของ `PolynomialSolver` สำหรับ degree ≥ 3  
จาก `src/algebra/solver/poly_solver.hpp`

## อัลกอริธึม

### ขั้นตอนรวม
1. **Normalize** → polynomial monic (หาร leading coefficient)
2. **Initial approximations**: กระจาย n จุดบน circle รัศมี r ในระนาบ complex
   - r = Cauchy's bound = 1 + max(|aₖ/aₙ|)
   - zₖ = r · exp(2πi·k / n) · offset (เพื่อ symmetry break)
3. **Durand-Kerner iterations** (≤ 2000 รอบ):
   - สำหรับ root แต่ละตัว i:
     - คำนวณ `delta_i = P(z_i) / ∏(z_i - z_j), j≠i`
     - อัปเดต `z_i := z_i - delta_i`
   - หยุดเมื่อ `max(|delta_i|) < tol × 1e-3`
4. **Newton polishing** (8 steps ต่อ root):
   - `z := z - P(z) / P'(z)`
5. **Classify**: จริง ถ้า |imag| ≤ tol × max(1, |real|)
6. **Merge**: รวม roots ที่ใกล้กัน (วัด multiplicity)

### Loop structure ที่ซับซ้อน
```
# Step 3: Durand-Kerner
for iteration in range(2000):                  ← outer: ≤ 2000 รอบ
    max_delta = 0
    for i in range(n):                         ← แต่ละ root
        Pz = eval_poly(coeffs, z[i])           ← O(n)
        denom = 1+0j
        for j in range(n):                     ← product ∏(z_i - z_j)
            if j != i:
                denom *= (z[i] - z[j])
        delta = Pz / denom
        new_z[i] = z[i] - delta
        max_delta = max(max_delta, |delta|)
    z = new_z
    if max_delta < tol * 1e-3: break           ← convergence

# Step 4: Newton
for i in range(n):                             ← แต่ละ root
    for _ in range(8):                         ← 8 Newton steps
        Pz  = eval_poly(coeffs, z[i])
        dPz = eval_deriv(coeffs, z[i])
        z[i] -= Pz / dPz
```

In [ ]:
from __future__ import annotations
import cmath
import math

# ──────────────────────────────────────────
# Helper: แทนค่า polynomial ด้วย Horner's method
# coeffs = [a_n, a_{n-1}, ..., a_1, a_0]  (highest degree first)
# ──────────────────────────────────────────

def eval_poly(coeffs: list[float], z: complex) -> complex:
    """Horner: P(z) = (...((a_n·z + a_{n-1})·z + ...)·z + a_0)"""
    result = complex(0)
    for c in coeffs:
        result = result * z + c
    return result


def eval_deriv(coeffs: list[float], z: complex) -> complex:
    """P'(z) โดยใช้ coefficients ที่ derived"""
    n = len(coeffs) - 1
    # derivative coeffs = [n·a_n, (n-1)·a_{n-1}, ..., 1·a_1]
    d_coeffs = [c * (n - i) for i, c in enumerate(coeffs[:-1])]
    if not d_coeffs:
        return complex(0)
    return eval_poly(d_coeffs, z)


def cauchy_bound(coeffs: list[float]) -> float:
    """รัศมีเริ่มต้น r = 1 + max(|a_k / a_n|) สำหรับ k < n"""
    an = coeffs[0]
    if abs(an) < 1e-300:
        return 1.0
    return 1.0 + max(abs(c / an) for c in coeffs[1:])


print("Helper functions loaded.")
# Quick sanity: P(x) = x^2 - 5x + 6 = (x-2)(x-3)
coeffs_test = [1.0, -5.0, 6.0]
assert abs(eval_poly(coeffs_test, 2.0)) < 1e-12
assert abs(eval_poly(coeffs_test, 3.0)) < 1e-12
print("eval_poly sanity OK")

In [ ]:
def durand_kerner(
    coeffs: list[float],
    tol: float = 1e-10,
    max_iter: int = 2000,
    verbose: bool = True,
    verbose_every: int = 1,   # แสดงทุก N iterations (ลดเสียง)
) -> list[complex]:
    """
    Durand-Kerner root finding

    Parameters
    ----------
    coeffs : list[float]
        สัมประสิทธิ์ polynomial เรียงจาก degree สูงสุด
        เช่น [1, -6, 11, -6] สำหรับ x³ - 6x² + 11x - 6
    tol : float
        เกณฑ์หยุด: max(|delta|) < tol * 1e-3
    max_iter : int
        จำนวน iterations สูงสุด
    verbose : bool
        แสดง step-by-step
    verbose_every : int
        แสดงรายละเอียดทุก N iterations

    Returns
    -------
    list[complex]
        approximations ของ roots ทั้งหมด
    """
    # ── Normalize: monic polynomial ─────────────────────────────
    an = coeffs[0]
    monic = [c / an for c in coeffs]
    n = len(monic) - 1          # degree

    if verbose:
        print("=" * 60)
        print("Durand-Kerner Method")
        print("=" * 60)
        print(f"  Original coeffs: {coeffs}")
        print(f"  Monic coeffs:    {[f'{c:.4g}' for c in monic]}")
        print(f"  Degree n = {n}")

    # ── Initial approximations: roots on circle ──────────────────
    r = cauchy_bound(monic)
    offset = complex(0.4, 0.9)   # ทำลาย symmetry เพื่อ convergence

    z = []
    for k in range(n):
        angle = 2 * math.pi * k / n
        zk = r * (offset ** k)   # approximation แบบ C++ ใช้
        z.append(zk)

    if verbose:
        print(f"\n  Cauchy bound r = {r:.4g}")
        print(f"  Initial approximations:")
        for k, zk in enumerate(z):
            print(f"    z[{k}] = {zk:.4g}")

    # ── Durand-Kerner iterations ─────────────────────────────────
    if verbose:
        print(f"\n  {'─'*50}")
        print(f"  Iterations (max={max_iter}, tol={tol:.1e}):")

    conv_iter = max_iter
    for iteration in range(max_iter):
        new_z = list(z)
        max_delta = 0.0

        show = verbose and (iteration % verbose_every == 0 or iteration < 3)
        if show:
            print(f"\n  === Iteration {iteration} ===")

        for i in range(n):
            # คำนวณ P(z[i])
            Pz = eval_poly(monic, z[i])

            # คำนวณ ∏(z[i] - z[j]) สำหรับ j ≠ i
            denom = complex(1, 0)
            for j in range(n):
                if j != i:
                    diff = z[i] - z[j]
                    denom *= diff

            if abs(denom) < 1e-300:
                delta = complex(0)
            else:
                delta = Pz / denom

            new_z[i] = z[i] - delta
            max_delta = max(max_delta, abs(delta))

            if show:
                print(
                    f"    i={i}: z={z[i]:.5g}  P(z)={Pz:.4g}"
                    f"  denom={denom:.4g}  delta={delta:.4g}"
                    f"  → new_z={new_z[i]:.5g}"
                )

        z = new_z
        if show:
            print(f"    max_delta = {max_delta:.4g}  (threshold = {tol*1e-3:.4g})")

        if max_delta < tol * 1e-3:
            conv_iter = iteration + 1
            if verbose:
                print(f"\n  ✓ Converged at iteration {conv_iter}")
            break

    if conv_iter == max_iter and verbose:
        print(f"  ⚠ Did not converge in {max_iter} iterations")

    if verbose:
        print(f"\n  After DK ({conv_iter} iterations):")
        for k, zk in enumerate(z):
            print(f"    z[{k}] = {zk:.8g}")

    return z

In [ ]:
def newton_polish(
    coeffs: list[float],
    z_approx: list[complex],
    steps: int = 8,
    verbose: bool = True,
) -> list[complex]:
    """
    Newton polishing: z := z - P(z)/P'(z) แต่ละ root, steps ครั้ง
    """
    an = coeffs[0]
    monic = [c / an for c in coeffs]
    polished = list(z_approx)

    if verbose:
        print("\n" + "=" * 60)
        print(f"Newton Polishing ({steps} steps per root)")
        print("=" * 60)

    for i, z in enumerate(z_approx):
        if verbose:
            print(f"  root {i}: start = {z:.8g}")

        for step in range(steps):
            Pz  = eval_poly(monic, z)
            dPz = eval_deriv(monic, z)

            if abs(dPz) < 1e-300:
                if verbose:
                    print(f"    step {step}: P'(z) ≈ 0, stop")
                break

            delta = Pz / dPz
            z_new = z - delta

            if verbose:
                print(
                    f"    step {step}: z={z:.6g}  P(z)={Pz:.4g}"
                    f"  P'(z)={dPz:.4g}  delta={delta:.4g}  → {z_new:.6g}"
                )

            z = z_new

        polished[i] = z
        if verbose:
            print(f"  root {i}: polished = {z:.10g}  |P(z)| = {abs(eval_poly(monic, z)):.4g}")

    return polished


def classify_roots(
    roots: list[complex],
    tol: float = 1e-10,
    verbose: bool = True,
) -> tuple[list[float], list[complex]]:
    """
    แยก roots เป็น real vs complex และ merge ที่ใกล้กัน

    Returns
    -------
    real_roots : list[float]  sorted ascending
    complex_roots : list[complex]
    """
    if verbose:
        print("\n" + "=" * 60)
        print("Classify Roots")
        print("=" * 60)

    real_roots_raw = []
    complex_roots = []

    for z in roots:
        re, im = z.real, z.imag
        threshold = tol * max(1.0, abs(re))
        is_real = abs(im) <= threshold

        if verbose:
            print(
                f"  z={z:.6g}  |im|={abs(im):.4g}  threshold={threshold:.4g}"
                f"  → {'REAL' if is_real else 'COMPLEX'}"
            )

        if is_real:
            real_roots_raw.append(re)
        else:
            complex_roots.append(z)

    # Merge near-duplicates
    real_roots_raw.sort()
    merged = []
    multiplicities = []

    for r in real_roots_raw:
        if merged and abs(r - merged[-1]) <= tol * max(1.0, abs(merged[-1])):
            multiplicities[-1] += 1
            if verbose:
                print(f"  MERGE {r:.6g} into {merged[-1]:.6g} (multiplicity {multiplicities[-1]})")
        else:
            merged.append(r)
            multiplicities.append(1)

    if verbose:
        print(f"\n  Real roots: {merged}")
        print(f"  Multiplicities: {multiplicities}")
        print(f"  Complex-only roots: {len(complex_roots)}")

    return merged, multiplicities, complex_roots

## Test Case 1: Cubic — x³ - 6x² + 11x - 6 = (x-1)(x-2)(x-3)

คาดหวัง: roots = {1, 2, 3} ทั้งหมด real

In [ ]:
# x^3 - 6x^2 + 11x - 6
coeffs1 = [1.0, -6.0, 11.0, -6.0]

z1 = durand_kerner(coeffs1, tol=1e-10, verbose=True, verbose_every=5)
z1p = newton_polish(coeffs1, z1, steps=8)
real1, mult1, cplx1 = classify_roots(z1p, tol=1e-10)

print(f"\nFinal real roots: {[f'{r:.6g}' for r in real1]}")
assert len(real1) == 3
for r, expected in zip(sorted(real1), [1.0, 2.0, 3.0]):
    assert abs(r - expected) < 1e-6, f"Expected {expected}, got {r}"
print("✓ Test 1 passed")

## Test Case 2: Quartic มี complex roots — x⁴ - 1 = (x-1)(x+1)(x-i)(x+i)

คาดหวัง: real roots = {-1, 1}, complex roots = {i, -i}

In [ ]:
# x^4 - 1
coeffs2 = [1.0, 0.0, 0.0, 0.0, -1.0]

z2 = durand_kerner(coeffs2, tol=1e-10, verbose=True, verbose_every=10)
z2p = newton_polish(coeffs2, z2, steps=8)
real2, mult2, cplx2 = classify_roots(z2p, tol=1e-8)

print(f"\nFinal real roots: {[f'{r:.6g}' for r in real2]}")
print(f"Complex roots: {[f'{z:.6g}' for z in cplx2]}")

assert sorted(round(r, 5) for r in real2) == [-1.0, 1.0]
assert len(cplx2) == 2
print("✓ Test 2 passed")

## Test Case 3: Repeated roots — (x-2)³ = x³ - 6x² + 12x - 8

คาดหวัง: root = 2 ที่มี multiplicity = 3  
Durand-Kerner ยังหาค่าได้ แต่ convergence ช้ากว่า

In [ ]:
# (x-2)^3 = x^3 - 6x^2 + 12x - 8
coeffs3 = [1.0, -6.0, 12.0, -8.0]

z3 = durand_kerner(coeffs3, tol=1e-10, max_iter=2000, verbose=True, verbose_every=50)
z3p = newton_polish(coeffs3, z3, steps=8)
real3, mult3, cplx3 = classify_roots(z3p, tol=1e-4)   # tol ผ่อนปรนสำหรับ repeated roots

print(f"\nFinal real roots: {[f'{r:.4g}' for r in real3]}")
print(f"Multiplicities: {mult3}")

# ทุก roots ควรใกล้ 2
for r in real3:
    assert abs(r - 2.0) < 0.01, f"Expected ~2.0, got {r}"
print("✓ Test 3 passed (repeated root at x=2)")

## Test Case 4: Degree 4 ทั้งหมด real — (x+1)(x+2)(x-3)(x-4)

= x⁴ - 4x³ - 11x² + 26x + 24

คาดหวัง: real roots = {-2, -1, 3, 4}

In [ ]:
# (x+1)(x+2)(x-3)(x-4) = x^4 - 4x^3 - 11x^2 + 26x + 24
coeffs4 = [1.0, -4.0, -11.0, 26.0, 24.0]

z4 = durand_kerner(coeffs4, tol=1e-10, verbose=True, verbose_every=20)
z4p = newton_polish(coeffs4, z4, steps=8)
real4, mult4, cplx4 = classify_roots(z4p, tol=1e-8)

print(f"\nFinal real roots: {[f'{r:.6g}' for r in sorted(real4)]}")
expected4 = [-2.0, -1.0, 3.0, 4.0]
for r, e in zip(sorted(real4), expected4):
    assert abs(r - e) < 1e-5, f"Expected {e}, got {r}"
print("✓ Test 4 passed")

## Convergence Visualization: delta ในแต่ละ iteration

วาดกราฟ max_delta vs iteration เพื่อดู convergence rate

In [ ]:
def durand_kerner_track(
    coeffs: list[float],
    tol: float = 1e-10,
    max_iter: int = 200,
) -> tuple[list[float], int]:
    """เหมือน durand_kerner แต่คืน history ของ max_delta"""
    an = coeffs[0]
    monic = [c / an for c in coeffs]
    n = len(monic) - 1

    r = cauchy_bound(monic)
    offset = complex(0.4, 0.9)
    z = [r * (offset ** k) for k in range(n)]

    deltas = []
    conv = max_iter

    for iteration in range(max_iter):
        new_z = list(z)
        max_delta = 0.0
        for i in range(n):
            Pz = eval_poly(monic, z[i])
            denom = complex(1)
            for j in range(n):
                if j != i:
                    denom *= (z[i] - z[j])
            if abs(denom) > 1e-300:
                delta = Pz / denom
                new_z[i] = z[i] - delta
                max_delta = max(max_delta, abs(delta))
        z = new_z
        deltas.append(max_delta)
        if max_delta < tol * 1e-3:
            conv = iteration + 1
            break

    return deltas, conv


# Track convergence สำหรับ polynomial ทั้ง 4 ตัว
polys = {
    "x³-6x²+11x-6  (3 distinct roots)": [1.0, -6.0, 11.0, -6.0],
    "x⁴-1 (2 real, 2 complex)": [1.0, 0.0, 0.0, 0.0, -1.0],
    "(x-2)³ (triple root)": [1.0, -6.0, 12.0, -8.0],
    "4 distinct roots": [1.0, -4.0, -11.0, 26.0, 24.0],
}

print(f"{'Polynomial':<35} {'Converged at iter':>18} {'Final delta':>14}")
print("-" * 70)
for name, c in polys.items():
    d, conv = durand_kerner_track(c)
    print(f"{name:<35} {'iter ' + str(conv):>18} {d[-1]:>14.4g}")

## สรุป Loop Structure

```
durand_kerner(coeffs):
│
├── normalize: monic = coeffs / coeffs[0]            ← O(n)
├── initial z[k] = r · offset^k  for k=0..n-1        ← O(n)
│
├── for iteration in range(2000):                     ← outer: ≤ 2000
│   ├── for i in range(n):                           ← แต่ละ root
│   │   ├── Pz = eval_poly(monic, z[i])              ← O(n) Horner
│   │   ├── for j in range(n) if j≠i:               ← O(n) product
│   │   │       denom *= (z[i] - z[j])
│   │   └── delta = Pz/denom; new_z[i] -= delta
│   ├── z = new_z
│   └── if max_delta < tol*1e-3: break               ← convergence
│   Total inner: O(n²) per iteration → O(2000·n²) worst case
│
newton_polish(coeffs, z, steps=8):
│   for i in range(n):                               ← แต่ละ root
│       for _ in range(8):                           ← 8 Newton steps
│           z[i] -= eval_poly(z[i]) / eval_deriv(z[i])   ← O(n) each
│
classify_roots(z, tol):
    for z in roots: check |imag| ≤ tol·max(1, |real|)
    sort + merge near-duplicates
```

| Test | Polynomial | Degree | Convergence | Real roots |
|------|-----------|--------|-------------|------------|
| 1 | (x-1)(x-2)(x-3) | 3 | เร็ว | 1, 2, 3 |
| 2 | x⁴-1 | 4 | เร็ว | ±1 (±i complex) |
| 3 | (x-2)³ | 3 | ช้า (repeated) | 2 (mult=3) |
| 4 | 4 distinct | 4 | เร็ว | -2,-1,3,4 |